# Group 3

In [ ]:
!pip install ftfy regex tqdm torchmetrics git+https://github.com/openai/CLIP.git

  Cloning https://github.com/openai/CLIP.git to /tmp/pip-req-build-vj2ymla_
  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-req-build-vj2ymla_
  Resolved https://github.com/openai/CLIP.git to commit d05afc436d78f1c48dc0dbf8e5980a9d471f35f6
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 40.4 MB/s eta 0:00:00


In [ ]:
import os
import clip
import torch
import torchmetrics
from tqdm import tqdm
from torch.utils.data import DataLoader
from torchvision.datasets import CIFAR10

# Define constants
LR = 1e-5 # Learning Rate
BATCH_SIZE = 128 # Number of images processed in one iteration
SHOTS = 5 # 5-way 5-shot
EPOCHS = 5 # Number of epochs

# Define controls
best_acc = 0.0 # Used to save best checkpoints

# Load the models
device = "cuda" if torch.cuda.is_available() else "cpu"
student, preprocess = clip.load('ViT-B/32', device, jit=False)
teacher, _ = clip.load('ViT-B/32', device)
for param in teacher.parameters():
    param.requires_grad = False

# Download the dataset
train_dataset = CIFAR10(root=os.path.expanduser("~/.cache"), download=True, train=True, transform=preprocess)
test_dataset =  CIFAR10(root=os.path.expanduser("~/.cache"), download=True, train=False, transform=preprocess)

# Prepare the data
train_dataloader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,       # Standard way to randomize
    drop_last=True      # Helpful for CLIP to keep matrix dimensions consistent
)
num_batches_train = len(train_dataloader)

all_class_texts = torch.cat([clip.tokenize(f"a photo of a {c}") for c in train_dataset.classes]).to(device)
NUM_CLASSES = len(train_dataset.classes)

test_dataloader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,       # Standard way to randomize
    drop_last=True      # Helpful for CLIP to keep matrix dimensions consistent
)
num_batches_test = len(test_dataloader)

# Prepare criterion and optimizer
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(student.parameters(), lr=LR, weight_decay=0.1)

# Prepare the training
for epoch in range(EPOCHS):
    print(f"Epoch: {epoch}")
    epoch_train_loss = 0
    student.train() # Set the model to training mode (enables dropout/batchnorm updates)

    print("Running Training...")
    for batch in tqdm(train_dataloader, total=num_batches_train):
        optimizer.zero_grad() # Clear previous gradients before starting a new optimization step

        # Unpack images and class indices
        # images shape: [Batch, 3, 224, 224]
        # labels shape: [Batch]
        images, class_ids = batch
        images = images.to(device)

        # Map class IDs to text descriptions
        # We clean class names (replace underscores with spaces) for better CLIP performance
        texts = [f"a photo of a {train_dataset.classes[i].replace('_', ' ')}" for i in class_ids]
        texts = clip.tokenize(texts).to(device)

        # Forward pass
        # CLIP computes similarity between all images and all texts in the batch
        logits_per_image, logits_per_text = student(images, texts)

        # Define Ground Truth
        # Creates a diagonal target [0, 1, ..., N-1] where image i matches text i
        ground_truth = torch.arange(len(images), dtype=torch.long, device=device)

        # Compute Symmetric Loss
        loss_i = criterion(logits_per_image, ground_truth)
        loss_t = criterion(logits_per_text, ground_truth)
        total_loss = (loss_i + loss_t) / 2

        # Optimization
        total_loss.backward() # Perform backpropagation to calculate gradients
        torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0) # This helps stability
        optimizer.step() # Update model weights based on calculated gradients

        epoch_train_loss += total_loss.item()

    print(f"Epoch: {epoch} | Training Loss: {epoch_train_loss:.4f}")

    student.eval() # Switch to eval mode (disables dropout)
    acc_top1_list = []
    acc_top5_list = []

    print("Running Evaluation...")
    for batch in tqdm(test_dataloader, total=num_batches_test):
        images, class_ids = batch
        images = images.to(device)
        class_ids = class_ids.to(device)

        with torch.no_grad():
            # Encode images and all possible classes
            image_features = student.encode_image(images)
            text_features = student.encode_text(all_class_texts)

            # Normalize features (CLIP works best with unit vectors)
            image_features /= image_features.norm(dim=-1, keepdim=True)
            text_features /= text_features.norm(dim=-1, keepdim=True)

            # Calculate cosine similarity
            # [Batch Size, 512] @ [512, 100] -> [Batch Size, 100]
            logits_per_image = (100.0 * image_features @ text_features.T).softmax(dim=-1)

            # Calculate Accuracy using torchmetrics
            acc_top1 = torchmetrics.functional.accuracy(logits_per_image, class_ids, task="multiclass", num_classes=NUM_CLASSES, top_k=1)
            acc_top5 = torchmetrics.functional.accuracy(logits_per_image, class_ids, task="multiclass", num_classes=NUM_CLASSES, top_k=5)

            acc_top1_list.append(acc_top1)
            acc_top5_list.append(acc_top5)

    # Final Mean Accuracy
    epoch_acc_top1 = torch.stack(acc_top1_list).mean()
    epoch_acc_top5 = torch.stack(acc_top5_list).mean()

    print(f"Test Top-1 Acc: {epoch_acc_top1:.2%}")
    print(f"Test Top-5 Acc: {epoch_acc_top5:.2%}")

    # Save best model
    if epoch_acc_top1 > best_acc:
        best_acc = epoch_acc_top1
        torch.save(student.state_dict(), "best_model.pt")
        print("New best model saved!")